# Simple RAG Pipeline

This section demonstrates a minimal working Retrieval-Augmented Generation (RAG) system.
We'll load text, chunk it, embed it, store in FAISS, retrieve, and generate an answer.

In [1]:
# Install dependencies
!pip install -q datasets faiss-cpu sentence-transformers transformers langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.5 MB/s eta 0:00:00


In [2]:
# Imports
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
from transformers import pipeline

In [3]:
docs = [
    "The economy is improving and job creation is up across sectors.",
    "SpaceX successfully launched another batch of satellites.",
    "The stock market closed higher today with gains in tech.",
    "Researchers found a new way to recycle plastics more efficiently.",
    "Climate change continues to cause extreme weather globally."
]

In [4]:
# Chunk documents (simple split)
def chunk_text(text, max_tokens=300):
    return [text[i:i+max_tokens] for i in range(0, len(text), max_tokens)]

In [5]:
chunks = []
for doc in docs:
    chunks.extend(chunk_text(doc))

In [6]:
# Embed with sentence transformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks, show_progress_bar=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# Store in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [8]:
# Retrieval function
def retrieve(query, top_k=5):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding), top_k)
    return [chunks[i] for i in I[0]]

In [9]:
# Load HF generator
generator = pipeline("text-generation", model="gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [10]:
# Query example
query = "What is happening with the stock market?"
retrieved = retrieve(query)

In [11]:
# Generate
context = " ".join(retrieved)
prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

response = generator(prompt, max_new_tokens=50)[0]["generated_text"]
print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Context: The stock market closed higher today with gains in tech. The economy is improving and job creation is up across sectors. Climate change continues to cause extreme weather globally. SpaceX successfully launched another batch of satellites. Researchers found a new way to recycle plastics more efficiently.

Question: What is happening with the stock market?

Answer:

The stock market has been steadily falling. As of today, it is expected to close lower than $1.00, down $1.80 and back to what it was in January. The market is now down $8.60.


# Advanced RAG Pipeline

Here we use LangChain and FAISS to build a modular and powerful RAG system.

In [12]:
# Install langchain for advanced RAG
!pip install -q langchain
!pip install -q -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.1/438.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00


In [13]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

In [14]:
# Create and split documents
with open("sample.txt", "w") as f:
    f.write("Data preprocessing is crucial in machine learning. It involves cleaning, normalizing, and transforming raw data into a usable format.")

loader = TextLoader("sample.txt")
docs = loader.load()

In [15]:
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
split_docs = splitter.split_documents(docs)

In [16]:
# Embeddings + Vectorstore
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(split_docs, embedding_model)

<ipython-input-16-4106295440>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [17]:
# Retriever
retriever = vectorstore.as_retriever()

In [18]:
# LLM setup
from transformers import pipeline as hf_pipeline
pipe = hf_pipeline("text-generation", model="gpt2", max_new_tokens=100)
llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cuda:0
<ipython-input-18-2973327037>:4: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [19]:
# RAG chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
result = qa_chain.run("What is the importance of data preprocessing?")
print(result)

<ipython-input-19-3594929284>:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain.run("What is the importance of data preprocessing?")
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Data preprocessing is crucial in machine learning.

and transforming raw data into a usable format.

learning. It involves cleaning, normalizing, and

Question: What is the importance of data preprocessing?
Helpful Answer: Preprocessing of data can be done and implemented as part of a program, but it's not the main point.

What is Machine Learning for?

In general, machine learning is used to train algorithms and to teach people how to do things. It's a way to teach people how to use data and algorithms to solve problems. It's also a way to teach people how to code, to write programs, and to learn new things.

Machine learning is the first step towards


# RAG Evaluation

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
# Define your test queries and reference answers for evaluation
test_queries = [
    "What is happening with the stock market?",
    "Tell me about recent SpaceX launches.",
    "Why is data preprocessing important?"
]

# Reference answers (gold standard)
reference_answers = [
    "The stock market closed higher today with gains in tech.",
    "SpaceX successfully launched another batch of satellites.",
    "Data preprocessing is crucial in machine learning because it cleans, normalizes, and transforms raw data."
]


In [22]:
# Function to generate answers using your generator + retriever pipeline
def generate_answer(query):
    retrieved_docs = retrieve(query, top_k=3)  # your retrieve function from above
    context = " ".join(retrieved_docs)
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    output = generator(prompt, max_new_tokens=50)[0]["generated_text"]
    # Extract answer part after 'Answer:' to avoid including prompt in eval
    answer = output.split("Answer:")[-1].strip()
    return answer

In [23]:
# Generate answers for all queries
generated_answers = [generate_answer(q) for q in test_queries]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [24]:
# Embed generated and reference answers
gen_embeds = model.encode(generated_answers)
ref_embeds = model.encode(reference_answers)

In [25]:
# Compute cosine similarity as evaluation metric
similarities = cosine_similarity(gen_embeds, ref_embeds)
scores = [similarities[i, i] for i in range(len(test_queries))]  # diagonal scores

In [26]:
# Print results
for i, query in enumerate(test_queries):
    print(f"Query: {query}")
    print(f"Generated Answer: {generated_answers[i]}")
    print(f"Reference Answer: {reference_answers[i]}")
    print(f"Semantic Similarity Score: {scores[i]:.4f}")
    print("-" * 60)

Query: What is happening with the stock market?
Generated Answer: The stock market is now up more than 17% in a year. The Dow Jones Industrial Average (DJIA), which measures the stock market's stock market performance, has already risen 17%. The S&P 500 (SPX), which measures the
Reference Answer: The stock market closed higher today with gains in tech.
Semantic Similarity Score: 0.4190
------------------------------------------------------------
Query: Tell me about recent SpaceX launches.
Generated Answer: In December, SpaceX launched a Falcon 9 rocket into orbit. On June 18, it was the first rocket to land on the International Space Station. On June 21, it was the first stage rocket to reach orbit and land aboard the International Space Station.
Reference Answer: SpaceX successfully launched another batch of satellites.
Semantic Similarity Score: 0.5375
------------------------------------------------------------
Query: Why is data preprocessing important?
Generated Answer: With smar